# RNN Model Univariate

In this section we implement the RNN Model (Recurrent Neural Network) using the **TimeSeriesDataset** approach with one-hot encoding.

The RNN (Recurrent Neural Network) Forecaster is a vanilla RNN model designed for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. It processes sequential data using recurrent connections that maintain a hidden state across time steps, allowing it to capture temporal dependencies.

**Layer Breakdown:**

- **RNN Layers:** 2 stacked RNN layers with Tanh activation
- **Hidden Size:** 64 units per layer
- **Dropout:** Applied between RNN layers (if >1 layer) and before final output
- **Output Layer:** Single fully connected layer producing 1-step forecast

**Advantages**

- **Simplicity:** Simpler architecture than LSTM/GRU (only hidden state, no cell state)
- **Speed:** Faster training due to fewer parameters
- **Sequential Processing:** Captures temporal dependencies in time series data

**Limitations**

- **Vanishing Gradients:** May struggle with long-term dependencies (sequences >10-20 steps)
- **Limited Memory:** No gating mechanisms to control information flow like LSTM/GRU


In [1]:
import torch
import torch.nn as nn

## Model

In [2]:
class RNNForecaster(nn.Module):
    """
    Vanilla RNN model for MULTIVARIATE time series forecasting.
    Architecture: RNN -> Dropout -> RNN -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    
    Simpler than LSTM - no cell state, only hidden state.
    Faster training but may struggle with long-term dependencies.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: RNN hidden dimension
            num_layers: Number of RNN layers
            dropout: Dropout rate
        """
        super(RNNForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # RNN layers (using Tanh activation by default)
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            nonlinearity='tanh'  # Can also use 'relu'
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # RNN forward pass
        # rnn_out: (batch_size, seq_length, hidden_size)
        # h_n: (num_layers, batch_size, hidden_size)
        rnn_out, h_n = self.rnn(x)
        
        # Take the output from the last time step
        last_output = rnn_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out

## Model Results without Exogenous Features


In this section, the RNN model is evaluated based on temporal features (value, year, and month) and one-hot encoded series identifiers, without the incorporation of external economic indicators. This baseline approach enables assessment of how well the model captures temporal dependencies and series-specific patterns using only historical information and temporal context. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|--------:|------------:|--------------:|----------:|----------:|
| 0 | 0.369208 | 128 | 0.10567 | 32 | 0.00013 | 2 | 25.38s |
| 1 | 0.381877 | 128 | 0.11781 | 128 | 0.00131 | 3 | 35.18s |
| 2 | 0.357050 | 64 | 0.31332 | 256 | 0.00034 | 3 | 59.86s |




###Best Hyperparameters

Validation Loss: 0.35705

Parameters:
  - learning_rate: 0.00034
  - batch_size: 64
  - num_layers: 3
  - hidden_size: 256
  - dropout: 0.31332

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/rnn/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 1 Results](./img/univariate/rnn/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 1 Results](./img/univariate/rnn/fold3/fold_results.png)

### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 90146.77 | 300.24 | 123.29 | 0.8384 | 68.94% |
| Fold 2 | 72547.65 | 269.35 | 116.39 | 0.8553 | 68.59% |
| Fold 3 | 68480.28 | 261.69 | 106.51 | 0.8696 | 64.13% |
| **Average** | **77058.23 ± 11088.68** | **277.10 ± 19.82** | **115.40 ± 8.50** | **0.8544 ± 0.0156** | **67.22% ± 2.51%** |

### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 9.2% ± 1.5% | 123 |
| 10-20% | 12.0% ± 0.8% | 160 |
| 20-30% | 13.6% ± 1.1% | 181 |
| 30-40% | 10.0% ± 0.3% | 133 |
| >40% | 55.3% ± 3.6% | 738 |


**Comparison with Baseline:**
The RNN univariate model with one-hot encoding achieves an average SMAPE of **67.22% ± 2.51%**, which is **4.96 percentage points lower** than the baseline 3-month rolling average (72.26% ± 7.06%). This indicates that the RNN model outperforms the baseline on average, demonstrating superior capability in capturing temporal dependencies.


## Model Results with Exogenous Features

In this section, the RNN model is evaluated based on temporal features (value, year, and month), one-hot encoded series identifiers, and external economic indicators. This approach enables assessment of how well the model captures series-specific patterns by leveraging both historical information and exogenous economic features. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.


### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|--------:|-----------:|--------------:|----------:|---------:|
| 0 | 0.39355 | 32 | 0.11509 | 32 | 0.00122 | 1 | 18.53s |
| 1 | 0.36077 | 32 | 0.26723 | 64 | 0.00098 | 2 | 33.27s |
| 2 | 0.37995 | 64 | 0.20047 | 256 | 0.00081 | 1 | 44.14s |





### Best Hyperparameters

Validation Loss: 0.36077

Parameters:
  - learning_rate: 0.00098
  - batch_size: 32
  - num_layers: 2
  - hidden_size: 64
  - dropout: 0.26723

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/rnn_exo/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/univariate/rnn_exo/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 3 Results](./img/univariate/rnn_exo/fold3/fold_results.png)

### Fold Results
| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 128578.29 | 358.58 | 147.06 | 0.7695 | 81.30% |
| Fold 2 | 88872.45 | 298.11 | 120.67 | 0.8227 | 73.16% |
| Fold 3 | 80924.97 | 284.47 | 106.19 | 0.8459 | 73.77% |
| **Average** | **99458.57 ± 26155.92** | **313.72 ± 41.32** | **124.64 ± 20.51** | **0.8127 ± 0.0386** | **76.08% ± 4.77%** |

### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 5.3% ± 2.3% | 70 |
| 10-20% | 8.8% ± 4.7% | 118 |
| 20-30% | 12.0% ± 4.4% | 161 |
| 30-40% | 9.5% ± 2.0% | 127 |
| >40% | 64.4% ± 10.2% | 859 |


**Comparison with Baseline:**

The RNN univariate model with exogenous features achieves an average SMAPE of **76.08% ± 4.77%**, which is **3.82 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). While the inclusion of exogenous economic indicators improved validation loss during hyperparameter tuning, the model still underperforms the baseline on average. However, the exogenous features reduced SMAPE variance across folds, indicating more stable predictions across different time periods.